# Data Cleaning
This notebook is used to apply the cleaning steps to the raw dataset and explain the reasons behind each transformation/decision.

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/naloxone.csv")

## Renaming Columns to snake_case
The original column names use title case with spaces, which makes them a bit harder to work with in Python. When it comes to coding I prefer to use snake_case naming convention. Renaming everything to snake_case is a small change that keeps the rest of the code cleaner. In this step I also fixed a typo in `Naxolone Administrations`. The correct spelling is *Na**lo**xone*.

In [23]:
df = df.rename(columns=lambda x: x.replace(" ", "_").lower())

In [24]:
df = df.rename(columns={
    "naxolone_administrations": "naloxone administrations"
})

## Dropping Unecessary Columns
Three columns were dropped because they don't add much analytical value to the analysis:
- `id`: just a composite key made from `incident_number` and `patient_number`.

- `neighbourhood`: has 223 unique values, many with only one incident, which makes it hard to analyze.
    - `ward` already does a great job covering this grouped geographical information into 15 categories.

- `neighbourhood_id`: just a code for each neighbourhood we already dropped. 

In [25]:
df = df.drop(columns=[
    "id",
    "neighbourhood",
    "neighbourhood_id"
])

## Expanding `dispatch_date`
`dispatch_date` was stored as a plain string (`"2021-08-08T04:33:22"`), which means pandas treats it as text. The best way to filter / extract information from this column is converting it to real datetime data type. After parsing it to datetime, I extracted `year`, `month`, `day_of_week`, and `hour` as separate columns. These are temporal dimensions columns that I plan to use to groupby/filter in the analysis. I also kept a `date` column (date only, no time) as a reference. The original `dispatch_date` was dropped after extracting the other columns out since the needed information lives in those new columns.

In [26]:
def expand_dispatch_date(df):
    """Parse dispatch_date and expand it into separate temporal columns.

    Converts the dispatch_date string column to datetime, extracts year,
    month, day_of_week, hour, and date into individual columns, then drops
    the original dispatch_date column.
    
    Parameters:
        df: DataFrame containing a dispatch_date column in datetime ISO format.
    
    Returns:
        DataFrame with dispatch_date replaced by the extracted columns.
    """

    df = df.copy() # don't modifies the original

    # parse to datetime
    df["dispatch_date"] = pd.to_datetime(df["dispatch_date"])

    # extract individual columns
    df["date"] = df["dispatch_date"].dt.date
    df["year"] = df["dispatch_date"].dt.year
    df["month"] = df["dispatch_date"].dt.month
    df["day_of_week"] = df["dispatch_date"].dt.day_of_week
    df["hour"] = df["dispatch_date"].dt.hour

    # drop now redundant, dispatch_date column
    df = df.drop(columns=["dispatch_date"])

    return df

In [27]:
df = expand_dispatch_date(df)

## Standardizing Missing Values
The `age` and `gender` columns uses both real NaN (missing values) and also the string `"Unknown"` as ways of showing that data is missing. This way, pandas is treating only the real NaN as missing values, so I replaced `"Unknown"` entries by real missing values.

In [28]:
# "age" column
df["age"] = df["age"].replace("Unknown", np.nan)

# "gender" column
df["gender"] = df["gender"].replace("Unknown", np.nan)

## Creating `age_midpoint`
`age` stores ranges like `"25 to 29"` instead of a single number, which is fine for categorical plots, but unusable when you need a numeric value to do calculations. I then created a new column, `age_midpoint`, to hold the mean (midpoint) between the lower and upper boundary of the range. Having both columns in the dataset is useful, so it covers both cases.

In [ ]:
def age_range_to_midpoint(age_range):
    """Convert a string age range to its midpoint as a number.
    
    Parameters:
        age_range: A string like "25 to 29", or NaN if missing

    Returns:
        The midpoint as a float, 100 for "Over 100",
        or NaN if it is a missing value.
    """
    if pd.isna(age_range):
        return np.nan

    parts = age_range.split(" ")
    lower = parts[0]
    upper = parts[-1]

    if lower == "Over": # handles the "Over 100" edge case
        return 100
    
    midpoint = (int(lower) + int(upper)) / 2

    return midpoint
 
df["age_midpoint"] = df["age"].apply(age_range_to_midpoint)